<a href="https://colab.research.google.com/github/zhangleino1/3d-crawl-toy/blob/main/minimax_h3_image_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax H3-Image on Google Colab (ComfyUI v0.45.0)

This session consumes about **10 compute units per hour** on a Colab G4 High-RAM runtime (figure from the source tutorial; Colab rates can change). Pick **Runtime → Change runtime type → GPU: G4, High RAM** before you run anything. The three weight files are 68.54 GB in total, so also confirm the runtime has enough disk.

This notebook follows the Colab / ComfyUI walkthrough in [阿磊 (@yangleiocx3)](https://x.com/yangleiocx3/status/2120312107124113490) — *MiniMax 4K 生图开源了，我把它装进了 96GB 云显卡* — and pins versions and file names to official sources checked on 2026-11-11:

- ComfyUI tutorial: https://docs.comfy.org/tutorials/image/minimax/minimax-h3-image
- Comfy-Org weights: https://huggingface.co/Comfy-Org/MiniMax-H3-Image
- Official model card: https://huggingface.co/MiniMaxAI/MiniMax-H3-Image
- License: https://huggingface.co/MiniMaxAI/MiniMax-H3-Image/blob/main/LICENSE (MiniMax-H3-Image Community License Agreement). The applicable territory excludes the EU, the UK, the Republic of Korea, and the United States. Hosted services or annualized revenue over USD 1 million need prior written authorization from MiniMax.

**What this notebook uses (article / 96 GB G4 setup), not the INT8 pack the official docs recommend for smaller GPUs:**

| File | Destination | Size | SHA256 |
| --- | --- | --- | --- |
| `minimax_h3_image_pruned_bf16.safetensors` | `ComfyUI/models/diffusion_models/` | 40,234,657,162 B (~40.23 GB) | `a767f5325db019588b3491a93d80288d336c826a7aeb87bec67675e554367222` |
| `qwen3vl_32b_minimax_h3_int8_convrot.safetensors` | `ComfyUI/models/text_encoders/` | 27,141,342,152 B (~27.14 GB) | `bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923` |
| `minimax_h3_image_vae_fp16.safetensors` | `ComfyUI/models/vae/` | 1,168,505,312 B (~1.17 GB) | `d80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af` |

Weights are downloaded from `Comfy-Org/MiniMax-H3-Image` at commit `892de7cb61df49753bf15145a711154803a355cb` (repo `main` tip when these hashes were read). The official ComfyUI text-to-image template ships the **INT8** DiT (`minimax_h3_image_pruned_int8_convrot.safetensors`, 21 GB) and an optional prompt expander; this notebook skips the expander and feeds a detailed English prompt straight to the model, which is how the official T2I template behaves when prompt expansion is off.

Run the cells **in order**. The last cell shuts the runtime down and is marked so you only run it by hand.


## 1. Check the GPU and disk

In [ ]:
# Expect a G4-class GPU with ~97 GB VRAM and a High-RAM machine (~176 GB RAM).
# The three weight files alone need 68.54 GB of disk.
!nvidia-smi
!echo "-----"
!free -h
!echo "-----"
!df -h /content /

# 1. 查看显卡型号、显存大小（VRAM）和驱动版本
!nvidia-smi

# 2. 查看系统内存（RAM）使用情况
!free -h

# 3. 查看 CPU 型号和核心数
!lscpu | grep -E "Model name|CPU\(s\):|Thread|Core"

# 4. 查看磁盘空间大小
!df -h /content /


Sun Nov 15 23:48:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   26C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone ComfyUI v0.45.0

In [ ]:
!rm -rf /tmp/ComfyUI_src
!git clone --branch v0.45.0 --depth 1 https://github.com/Comfy-Org/ComfyUI.git /tmp/ComfyUI_src
!cp -rn /tmp/ComfyUI_src/. /content/ComfyUI/
!rm -rf /tmp/ComfyUI_src
!/content/ComfyUI/venv/bin/pip install -r /content/ComfyUI/requirements.txt

%cd /content/ComfyUI
!git describe --tags --exact-match
!git rev-parse --short HEAD

Cloning into '/tmp/ComfyUI_src'...
remote: Enumerating objects: 1507, done.
remote: Counting objects: 100% (1507/1507), done.
remote: Compressing objects: 100% (1303/1303), done.
remote: Total 1507 (delta 159), reused 1129 (delta 155), pack-reused 0 (from 0)
Receiving objects: 100% (1507/1507), 12.57 MiB | 46.48 MiB/s, done.
Resolving deltas: 100% (159/159), done.
Note: switching to '989d93d0d968e704c589ce0146bb18c77f6b50cc'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

cp: warning: behavior of -n is non-portable and may c

## 3. Create a separate Python virtualenv

In [ ]:
# Official docs recommend an isolated environment so ComfyUI does not touch system Python.
from pathlib import Path
import shutil

VENV = Path("/content/ComfyUI/venv")

# 1. 强制清理之前创建失败的半成品 venv
if VENV.exists():
    shutil.rmtree(VENV)
    print(f"已清理损坏的环境: {VENV}")

# 2. 重新创建虚拟环境（带 --without-pip 避免 Python 3.13 的 ensurepip 报错）
!python3 -m venv --without-pip {VENV}

# 3. 手动下载并为该虚拟环境安装 pip
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!{VENV}/bin/python /tmp/get-pip.py

# 4. 验证 python 和 pip
!{VENV}/bin/python -V
!{VENV}/bin/pip -V


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.6 MB/s  0:00:00
Python 3.13.16
pip 26.3 from /content/ComfyUI/venv/lib/python3.13/site-packages/pip (python 3.13)


## 4. Install PyTorch (CUDA 13.0) and ComfyUI requirements

The install line is the official NVIDIA command from [Install ComfyUI Manually](https://docs.comfy.org/installation/manual_install):

```
pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
```

Then `pip install -r requirements.txt` (ComfyUI v0.45.0 pins `comfyui-frontend-package==1.58.5` and `comfyui-workflow-templates==0.11.105`).


In [ ]:
VENV_PIP = "/content/ComfyUI/venv/bin/pip"
VENV_PY = "/content/ComfyUI/venv/bin/python"

!{VENV_PIP} install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130
!{VENV_PIP} install -r /content/ComfyUI/requirements.txt
!{VENV_PIP} install -U "huggingface_hub[hf_transfer]" requests

!{VENV_PY} -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 127.8 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 116.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 120.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 99.0 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 115.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 145.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 112.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 147.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 MB 120.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 MB 127.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

\## 5. Download the three Comfy-Org weights at the pinned commit

Uses `huggingface_hub` with `hf_transfer` enabled. Files are placed in the folders the official docs and model card specify.


In [ ]:
import os
from pathlib import Path

# Enable the faster downloader provided by hf_transfer.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "Comfy-Org/MiniMax-H3-Image"
REVISION = "892de7cb61df49753bf15145a711154803a355cb"  # main tip as of 2026-11-10 16:04:39 UTC
MODELS_ROOT = Path("/content/ComfyUI/models")

WEIGHTS = [
    "diffusion_models/minimax_h3_image_pruned_bf16.safetensors",
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
    "vae/minimax_h3_image_vae_fp16.safetensors",
]

# Run the download inside the venv so hf_transfer is on the same interpreter.
download_script = r'''
import os
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
repo_id = %r
revision = %r
models_root = %r
weights = %r
for rel in weights:
    path = hf_hub_download(
        repo_id=repo_id,
        filename=rel,
        revision=revision,
        local_dir=models_root,
    )
    size = os.path.getsize(path)
    print(f"downloaded {rel} -> {path} ({size} bytes)")
''' % (REPO_ID, REVISION, str(MODELS_ROOT), WEIGHTS)

script_path = Path("/tmp/download_minimax_h3_weights.py")
script_path.write_text(download_script)
!/content/ComfyUI/venv/bin/python {script_path}


/content/ComfyUI/venv/lib/python3.13/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


downloaded diffusion_models/minimax_h3_image_pruned_bf16.safetensors -> /content/ComfyUI/models/diffusion_models/minimax_h3_image_pruned_bf16.safetensors (40234657162 bytes)


downloaded text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors -> /content/ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors (27141342152 bytes)


downloaded vae/minimax_h3_image_vae_fp16.safetensors -> /content/ComfyUI/models/vae/minimax_h3_image_vae_fp16.safetensors (1168505312 bytes)


## 6. Verify SHA256 of each weight file

In [ ]:
import hashlib
from pathlib import Path

EXPECTED = {
    "diffusion_models/minimax_h3_image_pruned_bf16.safetensors": {
        "sha256": "a767f5325db019588b3491a93d80288d336c826a7aeb87bec67675e554367222",
        "size": 40234657162,
    },
    "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors": {
        "sha256": "bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923",
        "size": 27141342152,
    },
    "vae/minimax_h3_image_vae_fp16.safetensors": {
        "sha256": "d80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af",
        "size": 1168505312,
    },
}

root = Path("/content/ComfyUI/models")
failed = False
for rel, meta in EXPECTED.items():
    path = root / rel
    size = path.stat().st_size
    if size != meta["size"]:
        failed = True
        print(f"SIZE MISMATCH {rel}: got {size}, expected {meta['size']}")
        continue
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    digest = h.hexdigest()
    ok = digest == meta["sha256"]
    failed = failed or not ok
    print(f"{'OK' if ok else 'HASH MISMATCH'} {rel}")
    print(f"  size={size} sha256={digest}")

if failed:
    raise SystemExit("One or more weight files failed verification. Stop here and re-download.")
print("All three files match the Hugging Face LFS SHA256 values.")


OK diffusion_models/minimax_h3_image_pruned_bf16.safetensors
  size=40234657162 sha256=a767f5325db019588b3491a93d80288d336c826a7aeb87bec67675e554367222
OK text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors
  size=27141342152 sha256=bc2ced0fbea64757fa9acddccfc0b3f4819d1dcf1da6c124d690d368be283923
OK vae/minimax_h3_image_vae_fp16.safetensors
  size=1168505312 sha256=d80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af
All three files match the Hugging Face LFS SHA256 values.


## 7. Start the ComfyUI server in the background

Official start command is `python main.py` ([manual install](https://docs.comfy.org/installation/manual_install)). Here it is bound to `127.0.0.1:8188` so the next cell can submit a prompt through the HTTP API.


In [ ]:
import os
import subprocess
import time
from pathlib import Path

import requests

COMFY_DIR = Path("/content/ComfyUI")
VENV_PY = COMFY_DIR / "venv" / "bin" / "python"
LOG = Path("/tmp/comfyui.log")
BASE = "http://127.0.0.1:8188"
LOCK = Path("/tmp/comfyui.pid")

def server_up():
    try:
        r = requests.get(f"{BASE}/system_stats", timeout=3)
        return r.status_code == 200
    except requests.RequestException:
        return False

if server_up():
    print("ComfyUI is already responding at 127.0.0.1:8188")
else:
    log_f = LOG.open("w")
    proc = subprocess.Popen(
        [str(VENV_PY), "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(COMFY_DIR),
        stdout=log_f,
        stderr=subprocess.STDOUT,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    LOCK.write_text(str(proc.pid))
    print(f"Started ComfyUI pid={proc.pid}, log={LOG}")
    deadline = time.time() + 180
    while time.time() < deadline:
        if proc.poll() is not None:
            print(LOG.read_text()[-4000:])
            raise SystemExit(f"ComfyUI exited early with code {proc.returncode}")
        if server_up():
            print("ComfyUI is ready at 127.0.0.1:8188")
            break
        time.sleep(2)
    else:
        print(LOG.read_text()[-4000:])
        raise SystemExit("Timed out waiting for ComfyUI to start. Check /tmp/comfyui.log")

print(requests.get(f"{BASE}/system_stats", timeout=10).json())


Started ComfyUI pid=8133, log=/tmp/comfyui.log
ComfyUI is ready at 127.0.0.1:8188
{'system': {'os': 'linux', 'ram_total': 189928583168, 'ram_free': 185631076352, 'comfyui_version': '0.45.0', 'required_frontend_version': '1.58.5', 'installed_templates_version': '0.11.105', 'required_templates_version': '0.11.105', 'comfy_package_versions': [{'name': 'comfyui-frontend-package', 'installed': '1.58.5', 'required': '1.58.5'}, {'name': 'comfyui-workflow-templates', 'installed': '0.11.105', 'required': '0.11.105'}, {'name': 'comfyui-embedded-docs', 'installed': '0.5.16', 'required': '0.5.16'}, {'name': 'comfy-aimdo', 'installed': '0.6.0', 'required': '0.6.0'}, {'name': 'comfy-kitchen', 'installed': '0.2.41', 'required': '0.2.41'}], 'python_version': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]', 'pytorch_version': '2.15.0+cu132', 'embedded_python': False, 'deploy_environment': 'local-git', 'argv': ['main.py', '--listen', '127.0.0.1', '--port', '8188']}, 'devices': [{'name': 'cuda:0 NVI

## 8. Run the 4096×4096 noodle-city test through the HTTP API

Graph (API format, node class names taken from the official T2I template `image_minimax_h3_image_t2i.json` and `comfy_extras/nodes_minimax_h3.py` in v0.45.0):

`UNETLoader` + `CLIPLoader` (`type=minimax`) + `VAELoader` → `MiniMaxH3ImageConditioning` → `KSampler` (`euler` / `simple` / 6 steps / CFG 1) with a `ConditioningZeroOut` negative (the official template does this; CFG 1 means there is no guidance pass) → `VAEDecode` → `SaveImage`.

The official template default is 4 MP so smaller cards do not OOM; this cell raises the canvas to the native 1:1 size of **4096×4096**. Prompt expansion is left off. To edit instead of generate, connect a reference image to `MiniMaxH3ImageConditioning` as `images.image_1` (and later images as `image_2` …).

The test prompt below is a detailed English caption written from the article's described scene (a bowl of instant noodles as a city). The exact wording in the X article was not independently retrieved, so this is not a verbatim quote.


In [17]:
import json
import shutil
import time
import uuid
from pathlib import Path

import requests
from IPython.display import Image, display

BASE = "http://127.0.0.1:8188"
OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPT = (
    "一位年轻美丽的亚洲女性，拥有白皙光滑的皮肤和精致的五官，长而直的深色头发自然垂落在肩膀上。她穿着一件设计大胆的黑色蕾丝性能比基尼 吊带丝袜，，凸显了全身的曲线，全身照，眼神平静而又点害羞地直视摄像机镜头。背景是高档酒店的室外游泳池，呈现出柔和的散景效果。画面采用中景构图，人物居中。光线为柔和的自然逆光，营造出梦幻、朦胧的氛围，色调以温暖的肤色、明亮的白色和对比鲜明的黑色为主。整体风格为超写实主义摄影，8k分辨率，极高画质，细腻的皮肤纹理，浅景深，电影级光影质感。"
)

# class_type and input names match the official template subgraph (UNETLoader, CLIPLoader type=minimax,
# VAELoader, MiniMaxH3ImageConditioning, ConditioningZeroOut, KSampler, VAEDecode, SaveImage).
workflow = {
    "1": {
        "class_type": "UNETLoader",
        "inputs": {
            "unet_name": "minimax_h3_image_pruned_bf16.safetensors",
            "weight_dtype": "default",
        },
    },
    "2": {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": "qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
            "type": "minimax",
            "device": "default",
        },
    },
    "3": {
        "class_type": "VAELoader",
        "inputs": {
            "vae_name": "minimax_h3_image_vae_fp16.safetensors",
        },
    },
    "4": {
        "class_type": "MiniMaxH3ImageConditioning",
        "inputs": {
            "clip": ["2", 0],
            "vae": ["3", 0],
            "prompt": PROMPT,
            "width": 4096,
            "height": 4096,
        },
    },
    "5": {
        "class_type": "ConditioningZeroOut",
        "inputs": {
            "conditioning": ["4", 0],
        },
    },
    "6": {
        "class_type": "KSampler",
        "inputs": {
            "model": ["1", 0],
            "positive": ["4", 0],
            "negative": ["5", 0],
            "latent_image": ["4", 1],
            "seed": 42,
            "steps": 6,
            "cfg": 1,
            "sampler_name": "euler",
            "scheduler": "simple",
            "denoise": 1,
        },
    },
    "7": {
        "class_type": "VAEDecode",
        "inputs": {
            "samples": ["6", 0],
            "vae": ["3", 0],
        },
    },
    "8": {
        "class_type": "SaveImage",
        "inputs": {
            "images": ["7", 0],
            "filename_prefix": "noodle_city_4096",
        },
    },
}

client_id = str(uuid.uuid4())
t0 = time.time()
resp = requests.post(
    f"{BASE}/prompt",
    json={"prompt": workflow, "client_id": client_id},
    timeout=60,
)
resp.raise_for_status()
payload = resp.json()
if "error" in payload or "node_errors" in payload and payload["node_errors"]:
    raise SystemExit(f"ComfyUI rejected the prompt: {json.dumps(payload, indent=2)}")
prompt_id = payload["prompt_id"]
print(f"queued prompt_id={prompt_id}")

# First run also loads ~68 GB of weights into VRAM; allow a long wait.
deadline = time.time() + 60 * 60
history = None
while time.time() < deadline:
    h = requests.get(f"{BASE}/history/{prompt_id}", timeout=30)
    h.raise_for_status()
    data = h.json()
    if prompt_id in data:
        status = data[prompt_id].get("status", {})
        if status.get("status_str") == "error" or status.get("completed") and status.get("status_str") != "success":
            raise SystemExit(f"ComfyUI job failed: {json.dumps(data[prompt_id], indent=2)[:8000]}")
        if data[prompt_id].get("outputs"):
            history = data[prompt_id]
            break
    time.sleep(5)
else:
    raise SystemExit("Timed out after 60 minutes waiting for the image. Check /tmp/comfyui.log")

elapsed = time.time() - t0
print(f"finished in {elapsed:.1f} s ({elapsed / 60:.2f} min)")

images = []
for node_out in history["outputs"].values():
    for img in node_out.get("images", []):
        images.append(img)

if not images:
    raise SystemExit(f"No images in history outputs: {json.dumps(history.get('outputs'), indent=2)}")

saved = []
for img in images:
    params = {
        "filename": img["filename"],
        "subfolder": img.get("subfolder", ""),
        "type": img.get("type", "output"),
    }
    raw = requests.get(f"{BASE}/view", params=params, timeout=120)
    raw.raise_for_status()
    dest = OUT_DIR / img["filename"]
    dest.write_bytes(raw.content)
    saved.append(dest)
    print(f"saved {dest} ({dest.stat().st_size} bytes)")
    display(Image(filename=str(dest), width=768))

# Also keep a copy next to ComfyUI's default output folder listing.
comfy_out = Path("/content/ComfyUI/output")
if comfy_out.exists():
    print("ComfyUI output dir:", sorted(p.name for p in comfy_out.glob("*"))[-10:])
print("Copies in", OUT_DIR, ":", [p.name for p in saved])


HTTPError: 400 Client Error: Bad Request for url: http://127.0.0.1:8188/prompt

# 9. 开始生成视频

In [14]:
# 删除纯图像扩散权重，保留共用的 qwen3vl_32b 文本编码器
! rm -f /content/ComfyUI/models/diffusion_models/minimax_h3_image_pruned_bf16.safetensors
! rm -f /content/ComfyUI/models/vae/minimax_h3_image_vae_fp16.safetensors
! /content/ComfyUI/venv/bin/pip cache purge
! df -h /content/

Files removed: 638 (1973.2 MB)
Directories removed: 1144
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   72G  165G  31% /


In [15]:
import os
from pathlib import Path

# 1. 清理残留缓存并查看当前可用空间（现在 Disk 应该已释放到 84GB 左右）
! rm -rf /tmp/h3_vid_dl /root/.cache/huggingface/hub/*
! df -h /content/

# 2. 关键修改：禁用触发 429 限流的 Xet 协议，改用标准 HTTPS 稳定下载
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

dl_h3_video_script = r'''
import os
import time
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_DISABLE_XET"] = "1"
models_root = "/content/ComfyUI/models"
repo_id = "Comfy-Org/MiniMax-H3"

targets = [
    ("diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors", "diffusion_models"),
    ("vae/minimax_h3_video_vae_fp16.safetensors", "vae"),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", "vae"),
]

for remote_path, subdir in targets:
    dest_dir = os.path.join(models_root, subdir)
    os.makedirs(dest_dir, exist_ok=True)
    local_name = os.path.basename(remote_path)
    final_path = os.path.join(dest_dir, local_name)
    if os.path.exists(final_path):
        print(f"已存在跳过: {final_path}")
        continue

    for attempt in range(1, 6):
        try:
            print(f"正在下载 {local_name} (第 {attempt} 次尝试)...")
            tmp_path = hf_hub_download(
                repo_id=repo_id,
                filename=remote_path,
                local_dir="/tmp/h3_vid_dl",
            )
            os.replace(tmp_path, final_path)
            os.system("rm -rf /tmp/h3_vid_dl/*")
            print(f"已就绪: {final_path} ({os.path.getsize(final_path) / 1e9:.2f} GB)")
            break
        except Exception as e:
            print(f"下载遇到波动: {e}，等待 15 秒后自动重试...")
            time.sleep(15)
    else:
        raise SystemExit(f"多次重试后仍无法下载 {remote_path}")
'''

Path("/tmp/download_minimax_h3_video.py").write_text(dl_h3_video_script)
! /content/ComfyUI/venv/bin/python /tmp/download_minimax_h3_video.py

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   72G  165G  31% /
/content/ComfyUI/venv/lib/python3.13/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
正在下载 minimax_h3_fl2va_pruned_fp8_scaled.safetensors (第 1 次尝试)...

已就绪: /content/ComfyUI/models/diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors (20.96 GB)
正在下载 minimax_h3_video_vae_fp16.safetensors (第 1 次尝试)...

已就绪: /content/ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors (5.21 GB)
正在下载 minimax_h3_audio_vae_fp32.safetensors (第 1 次尝试)...

已就绪: /content/ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors (0.61 GB)


In [16]:
import json
import shutil
import time
import uuid
from pathlib import Path
import requests
from IPython.display import Video, display

BASE = "http://127.0.0.1:8188"
COMFY_OUT = Path("/content/ComfyUI/output")
COMFY_IN = Path("/content/ComfyUI/input")
OUT_DIR = Path("/content/outputs")
COMFY_IN.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 找到第 8 步生成的图片并复制到 ComfyUI/input
candidates = sorted(COMFY_OUT.glob("noodle_city_4096_*.png")) or sorted(OUT_DIR.glob("noodle_city_4096_*.png"))
if not candidates:
    raise SystemExit("未找到第 8 步生成的图片！请先运行第 8 步生成图片。")

src_img = candidates[-1]
input_name = src_img.name
shutil.copy2(src_img, COMFY_IN / input_name)
print(f"已绑定首帧图片: {input_name}")

# 2. 获取已下载的 H3 视频模型与 VAE
diff_models = [p.name for p in Path("/content/ComfyUI/models/diffusion_models").glob("*fl2va*.safetensors")]
vae_models = [p.name for p in Path("/content/ComfyUI/models/vae").glob("*h3*.safetensors")]
video_vae = next((v for v in vae_models if "video" in v.lower()), None)

if not diff_models or not video_vae:
    raise SystemExit(f"缺少必要的视频模型文件: diff={diff_models}, vae={vae_models}")

# 3. 10秒连贯动作提示词：把交叉的腿放下来 + 露出自然微笑
VIDEO_PROMPT = (
    "基于首帧画面，站在窗边的年轻亚洲女性开始自然地舒展身体。"
    "起初她原本交叉的双腿缓缓地放下来，双脚平稳自然地站立在地板上，身体姿态变得放松而优雅。"
    "紧接着，她微微调整姿势，保持一只手轻扶白色半透明窗帘，视线温柔地直视镜头，嘴角缓缓扬起，露出甜美、自然而又略带羞涩的微笑。"
    "微风轻拂她的黑色长发和半透明窗帘，柔和的自然阳光洒在她的脸庞和黑色连体泳衣上。"
    "电影级写实摄影，镜头保持稳定平滑，人物五官、身材比例与服装细节始终与首帧高度一致，光影细腻流畅。"
)

# 4. 生成 10 秒视频（241 帧 @ 24fps = 10.04 秒，满足 8n+1 时序步长）
workflow = {
    "1": {
        "class_type": "UNETLoader",
        "inputs": {"unet_name": diff_models[0], "weight_dtype": "default"},
    },
    "2": {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": "qwen3vl_32b_minimax_h3_int8_convrot.safetensors",
            "type": "minimax",
            "device": "default",
        },
    },
    "3": {
        "class_type": "VAELoader",
        "inputs": {"vae_name": video_vae},
    },
    "10": {
        "class_type": "LoadImage",
        "inputs": {"image": input_name},
    },
    "4": {
        "class_type": "MiniMaxH3ImageToVideo",
        "inputs": {
            "clip": ["2", 0],
            "vae": ["3", 0],
            "prompt": VIDEO_PROMPT,
            "width": 768,
            "height": 768,
            "length": 241,  # 241 帧 @ 24fps ≈ 10 秒
            "first_frame": ["10", 0],
        },
    },
    "5": {
        "class_type": "ConditioningZeroOut",
        "inputs": {"conditioning": ["4", 0]},
    },
    "6": {
        "class_type": "KSampler",
        "inputs": {
            "model": ["1", 0],
            "positive": ["4", 0],
            "negative": ["5", 0],
            "latent_image": ["4", 1],
            "seed": 2026,
            "steps": 20,
            "cfg": 1.0,
            "sampler_name": "euler",
            "scheduler": "simple",
            "denoise": 1.0,
        },
    },
    "7": {
        "class_type": "VAEDecode",
        "inputs": {"samples": ["6", 0], "vae": ["3", 0]},
    },
    "8": {
        "class_type": "CreateVideo",
        "inputs": {"images": ["7", 0], "fps": 24.0},
    },
    "11": {
        "class_type": "SaveVideo",
        "inputs": {
            "video": ["8", 0],
            "filename_prefix": "minimax_h3_i2v_10s",
            "format": "mp4",
            "codec": "h264",
        },
    },
}

client_id = str(uuid.uuid4())
t0 = time.time()
resp = requests.post(f"{BASE}/prompt", json={"prompt": workflow, "client_id": client_id}, timeout=60)
payload = resp.json()
if "error" in payload or payload.get("node_errors"):
    raise SystemExit(f"节点校验报错:\n{json.dumps(payload, indent=2, ensure_ascii=False)}")

prompt_id = payload["prompt_id"]
print(f"10秒图生视频任务已提交: prompt_id={prompt_id}（241帧，预计耗时 3~4 分钟）...")

deadline = time.time() + 3600
history = None
while time.time() < deadline:
    data = requests.get(f"{BASE}/history/{prompt_id}", timeout=30).json()
    if prompt_id in data:
        status = data[prompt_id].get("status", {})
        if status.get("status_str") == "error":
            raise SystemExit(f"视频生成报错:\n{json.dumps(data[prompt_id], indent=2, ensure_ascii=False)[:4000]}")
        if data[prompt_id].get("outputs"):
            history = data[prompt_id]
            break
    time.sleep(5)

print(f"生成完成！耗时: {time.time() - t0:.1f} 秒")

for node_out in history["outputs"].values():
    for vid in node_out.get("images", []) + node_out.get("videos", []):
        params = {
            "filename": vid["filename"],
            "subfolder": vid.get("subfolder", ""),
            "type": vid.get("type", "output"),
        }
        raw = requests.get(f"{BASE}/view", params=params, timeout=120)
        dest = OUT_DIR / vid["filename"]
        dest.write_bytes(raw.content)
        print(f"视频已保存至: {dest} ({dest.stat().st_size / 1e6:.2f} MB)")
        display(Video(str(dest), embed=True, width=512))

已绑定首帧图片: noodle_city_4096_00004_.png
10秒图生视频任务已提交: prompt_id=cfe9a15d-cf89-41eb-be9c-0c71c7e0b8f2（241帧，预计耗时 3~4 分钟）...
生成完成！耗时: 460.2 秒
视频已保存至: /content/outputs/minimax_h3_i2v_10s_00001_.mp4 (2.31 MB)


## 10. Shut down the runtime (manual only)

**Do not run this cell automatically.** It calls `google.colab.runtime.unassign()`, which ends the session and stops the G4 billing clock. Run it only after you have downloaded `/content/outputs`.


In [18]:
# MANUAL ONLY — do not run automatically. This ends the Colab runtime and stops billing.
from google.colab import runtime
runtime.unassign()
